# 38 — DiD: Publication & Citation Trajectories Around Award Year

## Goal
Visualise how **publications per year** and **citations received per year** evolve for award-winning authors  
in a window relative to their award year (`t = 0`).  
We compare **Junior** (career age < 5 at award time) vs **Senior** authors.

## Design decisions
| Decision | Choice | Rationale |
|----------|--------|-----------|
| Unit of analysis | **One row per unique author** (deduplication) | Authors on multiple award papers would otherwise be counted multiple times and skew the trajectory |
| Citation metric | **Citations received per year** (not cumulative) | Shows the *flow* of attention; cumulative confounds early- vs late-career starting levels |
| Aggregation | **Median** per relative year × group | Robust to the heavy-tailed distributions typical in bibliometrics |
| Window | **±5 years** where available; for junior authors the pre-award window is capped at their career age (e.g. career_age=3 → only −3…0 for pre-award) | Avoids fabricating data for years before the author published anything |
| Breakdowns | 1. Overall (junior vs senior) · 2. Split by conference · 3. Split by award type | Allows us to check whether the signal is driven by a single venue or award category |

## Input
`../data/raw/icwsm_jcdl_author_profiles.csv` — built in notebook 36.

## Output
- `../data/processed/author_yearly_trajectories.csv` — long-format per-author yearly counts
- Plots saved to `../data/processed/`


## 0. Imports & helpers

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

OPENALEX_EMAIL = 'sherotowshaw@gmail.com'
HEADERS = {'User-Agent': f'thesis-research mailto:{OPENALEX_EMAIL}'}
BASE = 'https://api.openalex.org'
SLEEP = 0.12

os.makedirs('../data/processed', exist_ok=True)

def oa_get(url, params=None):
    r = requests.get(url, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    time.sleep(SLEEP)
    return r.json()


## 1. Load author profiles & deduplicate

> **Deduplication rule:** an author may appear on more than one award paper (same or different conference/year).  
> We keep only **one record per unique `author_id`**, choosing the *earliest* award year so the pre-award  
> window is as long as possible. This ensures each author contributes exactly one trajectory.

In [ ]:
profiles = pd.read_csv('../data/raw/icwsm_jcdl_author_profiles.csv')

print(f'Author-paper rows before dedup : {len(profiles)}')
print(f'Unique author IDs              : {profiles["author_id"].nunique()}')

profiles = profiles[~profiles['career_age_missing']].copy()

dedup = (
    profiles
    .sort_values('award_year')
    .drop_duplicates(subset='author_id', keep='first')
    .reset_index(drop=True)
)

print(f'After dedup (unique authors)   : {len(dedup)}')
print(f'Junior share                   : {dedup["is_junior"].mean():.1%}')
dedup[['author_name','conference','award_type','award_year','career_age','is_junior']].head(8)


## 2. Fetch per-year counts from OpenAlex

For each unique author we call `/authors/{id}` and extract `counts_by_year`:  
a list of `{year, works_count, cited_by_count}` objects covering all years the API tracks.

> `cited_by_count` here is **citations received in that calendar year** — exactly the per-year flow metric we want.

Results are cached to `../data/processed/author_yearly_raw.csv` so the API is only hit once.

In [ ]:
CACHE_PATH = '../data/processed/author_yearly_raw.csv'

if os.path.exists(CACHE_PATH):
    yearly_raw = pd.read_csv(CACHE_PATH)
    print(f'Loaded from cache: {len(yearly_raw)} rows')
else:
    records = []
    for i, row in dedup.iterrows():
        aid = str(row['author_id'])
        if not aid.startswith('https://openalex.org/'):
            continue
        try:
            data = oa_get(f"{BASE}/authors/{aid.split('/')[-1]}")
        except Exception as e:
            print(f'  ERROR {aid}: {e}')
            continue
        for entry in data.get('counts_by_year', []):
            records.append({
                'author_id'     : aid,
                'year'          : entry['year'],
                'works_count'   : entry.get('works_count', 0),
                'cited_by_count': entry.get('cited_by_count', 0),
            })
        if (i + 1) % 50 == 0:
            print(f'  fetched {i+1}/{len(dedup)}')

    yearly_raw = pd.DataFrame(records)
    yearly_raw.to_csv(CACHE_PATH, index=False)
    print(f'Fetched & cached: {len(yearly_raw)} rows')


## 3. Build relative-year trajectories

For each author we compute `relative_year = calendar_year − award_year`, so `t = 0` is the award year.

### Window capping for junior authors
A junior author with `career_age = 2` only has 2 years of pre-award data; going back to `t = −5` would  
include years before they published anything (all zeros, which would artificially suppress the pre-award median).  
We therefore **drop relative years where `relative_year < −career_age`** for each author.  
The post-award window is capped at +5 or the latest year available in the data, whichever is smaller.

In [ ]:
MAX_WINDOW = 5

merged = yearly_raw.merge(
    dedup[['author_id','award_year','career_age','is_junior','conference','award_type']],
    on='author_id', how='inner'
)

merged['relative_year'] = merged['year'] - merged['award_year']

merged = merged[
    (merged['relative_year'] <= MAX_WINDOW) &
    (merged['relative_year'] >= -merged['career_age'].clip(upper=MAX_WINDOW))
].copy()

merged['seniority'] = merged['is_junior'].map({True: 'Junior (<5 yr)', False: 'Senior (≥5 yr)'})

print(f'Trajectory rows after windowing: {len(merged)}')
print(f'Unique authors in trajectories : {merged["author_id"].nunique()}')
merged.head()

merged.to_csv('../data/processed/author_yearly_trajectories.csv', index=False)
print('Saved → ../data/processed/author_yearly_trajectories.csv')


## 4. Helper: DiD line-plot function

Reusable function that takes a filtered dataframe and produces a two-panel figure  
(publications | citations) with one line per seniority group.

In [ ]:
COLORS = {'Junior (<5 yr)': '#E07B39', 'Senior (≥5 yr)': '#3A7EBB'}

def did_plot(df, title, savepath):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

    for metric, ax, ylabel, panel_title in [
        ('works_count',    axes[0], 'Median publications per year', 'Publications per year'),
        ('cited_by_count', axes[1], 'Median citations received per year', 'Citations received per year'),
    ]:
        for group, grp_df in df.groupby('seniority'):
            agg = (
                grp_df.groupby('relative_year')[metric]
                      .median()
                      .reset_index()
                      .sort_values('relative_year')
            )
            ax.plot(
                agg['relative_year'], agg[metric],
                marker='o', markersize=5,
                color=COLORS.get(group, 'gray'),
                label=group
            )

        ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6, label='Award year (t=0)')
        ax.set_xlabel('Years relative to award')
        ax.set_ylabel(ylabel)
        ax.set_title(panel_title)
        ax.legend(fontsize=9)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig(savepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {savepath}')


## 5. Overall DiD (all venues & award types combined)

Baseline view: junior vs senior across the full dataset.

In [ ]:
did_plot(
    df=merged,
    title='DiD — All venues & award types (Junior vs Senior)',
    savepath='../data/processed/did_overall.png'
)


## 6. DiD split by Conference

One figure per conference (ICWSM and JCDL) to check whether the trajectory pattern is venue-specific.

> **Note:** ICWSM has fewer observations than JCDL, so the median lines may be noisier.

In [ ]:
for conf, grp in merged.groupby('conference'):
    n_authors = grp['author_id'].nunique()
    print(f'\n{conf}: {n_authors} unique authors')
    did_plot(
        df=grp,
        title=f'DiD — {conf} (Junior vs Senior, n={n_authors} authors)',
        savepath=f'../data/processed/did_conf_{conf.lower()}.png'
    )


## 7. DiD split by Award Type

One figure per award type. Award types with **fewer than 5 unique authors** in either the junior or senior  
group are skipped — the median from very small groups is not informative.

> Award types with sufficient data in both groups will be visualised; others are reported in a summary table.

In [ ]:
MIN_AUTHORS_PER_GROUP = 5

skipped = []
for award, grp in merged.groupby('award_type'):
    counts = grp.drop_duplicates('author_id')['is_junior'].value_counts()
    n_junior = counts.get(True, 0)
    n_senior = counts.get(False, 0)
    if n_junior < MIN_AUTHORS_PER_GROUP or n_senior < MIN_AUTHORS_PER_GROUP:
        skipped.append({'award_type': award, 'n_junior': n_junior, 'n_senior': n_senior})
        continue
    n_total = grp['author_id'].nunique()
    safe_name = award.lower().replace(' ', '_').replace('/', '_')
    did_plot(
        df=grp,
        title=f'DiD — {award} (Junior vs Senior, n={n_total} authors)',
        savepath=f'../data/processed/did_award_{safe_name}.png'
    )

if skipped:
    print('\n--- Award types skipped (too few authors in at least one group) ---')
    print(pd.DataFrame(skipped).to_string(index=False))


## 8. Coverage summary

How many authors have data at each relative year? Important for interpreting the tails of the window:  
fewer authors contribute at extreme relative years (especially far pre-award for junior authors).

In [ ]:
coverage = (
    merged
    .groupby(['relative_year', 'seniority'])['author_id']
    .nunique()
    .reset_index(name='n_authors')
    .pivot(index='relative_year', columns='seniority', values='n_authors')
    .fillna(0)
    .astype(int)
)

print('Authors contributing to each relative year:')
print(coverage.to_string())
